# Ноутбук 01: Построение базы знаний для RAG-эксперимента

**Цель:** подготовить инфраструктуру для исследования устойчивости RAG-агентов к зашумлённым данным.

## Что делает этот ноутбук

1. Подключается к Qdrant Cloud и **очищает все существующие коллекции**.
2. Загружает **HotpotQA (distractor split)** — датасет мультихоп-вопросов с готовыми gold-параграфами и дистракторами.
3. Отбирает **100 вопросов** со стратификацией по типу (bridge/comparison) и сложности (easy/medium/hard) для пилотного эксперимента.
4. Формирует **корпус документов**: gold-параграфы целевых вопросов + distractor-параграфы + фоновые параграфы из других вопросов.
5. Индексирует корпус в коллекцию `rag_clean` в Qdrant с использованием FastEmbed (модель `BAAI/bge-small-en-v1.5`).
6. Сохраняет артефакты эксперимента (вопросы, gold-mapping, метаданные корпуса) на локальный диск.

## Научный дизайн

**Два режима эксперимента**, под которые готовится база:

- **Controlled-context** (oracle + noise): золотой контекст всегда в промпте, к нему подмешивается шум. Измеряет устойчивость **генератора** к шуму в контексте.
- **End-to-end**: будет создана вторая коллекция `rag_noisy` с внедрённым шумом в базу, retriever работает обычно. Измеряет устойчивость **всего пайплайна**.

В этом ноутбуке готовим **только чистую базу** и фиксированный набор вопросов. Загрязнение и noisy-коллекция — в следующем ноутбуке.

## Шаг 0. Установка зависимостей и инициализация

`fastembed` даёт CPU-embeddings без VRAM (экономим под LLM в Colab).
`qdrant-client` — для работы с облачным Qdrant.
`datasets` — для загрузки HotpotQA с HuggingFace.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q qdrant-client fastembed datasets tqdm langchain-text-splitters pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 39.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")

assert QDRANT_URL, "Не найден QDRANT_URL"
assert QDRANT_API_KEY, "Не найден QDRANT_API_KEY"

# Параметры эксперимента
SEED = 42
N_QUESTIONS = 100  # пилотный блок
BACKGROUND_QUESTIONS = 400  # дополнительные вопросы, их параграфы пойдут в корпус как фон для реалистичного retrieval

# Параметры индексации
COLLECTION_CLEAN = "rag_clean"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"  # 384 dim, CPU-friendly
EMBEDDING_DIM = 384
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

# Директория для артефактов
ARTIFACTS_DIR = Path("/content/drive/MyDrive/rag_experiment/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print(f"Artifacts dir: {ARTIFACTS_DIR.absolute()}")
print(f"Seed: {SEED}")
print(f"Target questions: {N_QUESTIONS}")

Artifacts dir: /content/drive/MyDrive/rag_experiment/artifacts
Seed: 42
Target questions: 100


## Шаг 1. Подключение к Qdrant и очистка всех коллекций

Удаляем всё, что уже лежит в облачном Qdrant, чтобы стартовать с чистого состояния.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# Получаем список всех коллекций и удаляем
existing = qdrant.get_collections().collections
print(f"Найдено коллекций в Qdrant: {len(existing)}")
for col in existing:
    print(f"  - удаляю: {col.name}")
    qdrant.delete_collection(collection_name=col.name)

# Проверка
remaining = qdrant.get_collections().collections
print(f"\nПосле очистки коллекций: {len(remaining)}")
assert len(remaining) == 0, "Qdrant не очищен до конца"

Найдено коллекций в Qdrant: 1
  - удаляю: rag_clean

После очистки коллекций: 0


## Шаг 2. Загрузка HotpotQA (distractor)

**Почему distractor split?**

В distractor-версии для каждого вопроса уже есть 10 параграфов: 2 gold (`supporting_facts`) и 8 related-but-irrelevant distractor'ов. Это даёт нам:

- готовую разметку gold-документов для controlled-context режима;
- готовые семантически близкие дистракторы (тип шума «semantic distractors» из классификации в теории, раздел 3.1);
- разнообразие типов вопросов (bridge / comparison) и сложности (easy / medium / hard).

Используем `validation` split — он не использовался для обучения retriever-моделей, что исключает data leakage.

In [ ]:
# Загрузка (~46 MB распакованных)
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation", trust_remote_code=True)
print(f"Всего примеров в validation: {len(ds)}")
print(f"\nПример записи:")
sample = ds[0]
print(f"  id: {sample['id']}")
print(f"  question: {sample['question']}")
print(f"  answer: {sample['answer']}")
print(f"  type: {sample['type']}")
print(f"  level: {sample['level']}")
print(f"  n_paragraphs: {len(sample['context']['title'])}")
print(f"  supporting titles: {sample['supporting_facts']['title']}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpotqa/hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpotqa/hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to re

README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Всего примеров в validation: 7405

Пример записи:
  id: 5a8b57f25542995d1e6f1371
  question: Were Scott Derrickson and Ed Wood of the same nationality?
  answer: yes
  type: comparison
  level: hard
  n_paragraphs: 10
  supporting titles: ['Scott Derrickson', 'Ed Wood']


In [ ]:
# Распределение по типам и сложности
types = Counter(ds["type"])
levels = Counter(ds["level"])
print("По типам:", dict(types))
print("По сложности:", dict(levels))

По типам: {'comparison': 1487, 'bridge': 5918}
По сложности: {'hard': 7405}


## Шаг 3. Стратифицированный отбор 100 вопросов

**Стратегия отбора:** равномерно по пересечению {type} × {level}, фильтруя некорректные записи.

**Критерии фильтрации:**
- ответ непуст и короче 100 символов (иначе это не факт, а мини-эссе);
- supporting_facts указывают ровно на **2 разных title** (настоящий мультихоп);
- все supporting titles присутствуют в `context.title` (отбрасываем битые записи);
- параграф gold-документа не слишком короткий (sum(sentences) >= 100 символов).

Это даёт нам чистую пилотную выборку, где gold-документы гарантированно содержат ответ.

In [ ]:
def normalize_example(ex):
    """Превращает пример HotpotQA в удобный словарь. Возвращает None, если запись плохая."""
    answer = (ex["answer"] or "").strip()
    if not answer or len(answer) > 100:
        return None

    titles = ex["context"]["title"]
    sentences = ex["context"]["sentences"]
    if len(titles) != len(sentences):
        return None

    # title -> полный текст параграфа
    title_to_text = {}
    for t, sents in zip(titles, sentences):
        text = " ".join(sents).strip()
        if text:
            title_to_text[t] = text

    support_titles = list(dict.fromkeys(ex["supporting_facts"]["title"]))  # уникальные, с сохранением порядка
    if len(support_titles) != 2:
        return None
    if not all(t in title_to_text for t in support_titles):
        return None
    if any(len(title_to_text[t]) < 100 for t in support_titles):
        return None

    return {
        "id": ex["id"],
        "question": ex["question"],
        "answer": answer,
        "type": ex["type"],
        "level": ex["level"],
        "gold_titles": support_titles,
        "all_titles": list(title_to_text.keys()),
        "paragraphs": title_to_text,  # dict: title -> text
    }

# Фильтрация
clean_examples = []
for ex in tqdm(ds, desc="Фильтрация"):
    norm = normalize_example(ex)
    if norm is not None:
        clean_examples.append(norm)

print(f"После фильтрации: {len(clean_examples)} из {len(ds)}")
print("Типы после фильтрации:", Counter(e["type"] for e in clean_examples))
print("Уровни после фильтрации:", Counter(e["level"] for e in clean_examples))

Фильтрация:   0%|          | 0/7405 [00:00<?, ?it/s]

После фильтрации: 7149 из 7405
Типы после фильтрации: Counter({'bridge': 5773, 'comparison': 1376})
Уровни после фильтрации: Counter({'hard': 7149})


In [ ]:
# Стратифицированный сэмплинг
# Идея: равные доли по (type × level), всего 2×3=6 страт → ~17 вопросов в страте для 100 всего

strata = defaultdict(list)
for ex in clean_examples:
    key = (ex["type"], ex["level"])
    strata[key].append(ex)

print("Размер каждой страты:")
for key, items in strata.items():
    print(f"  {key}: {len(items)}")

# Распределяем N_QUESTIONS пропорционально, но не меньше 10 на страту если возможно
n_strata = len(strata)
base_per_stratum = N_QUESTIONS // n_strata
remainder = N_QUESTIONS - base_per_stratum * n_strata

selected = []
strata_keys = sorted(strata.keys())
for i, key in enumerate(strata_keys):
    take = base_per_stratum + (1 if i < remainder else 0)
    items = strata[key]
    if len(items) < take:
        print(f"  ПРЕДУПРЕЖДЕНИЕ: в страте {key} только {len(items)} примеров, беру все")
        take = len(items)
    rng = random.Random(SEED + i)  # детерминированность на уровне страт
    picked = rng.sample(items, take)
    selected.extend(picked)

print(f"\nОтобрано вопросов: {len(selected)}")
print("Финальное распределение по стратам:")
final_dist = Counter((e["type"], e["level"]) for e in selected)
for k, v in sorted(final_dist.items()):
    print(f"  {k}: {v}")

Размер каждой страты:
  ('comparison', 'hard'): 1376
  ('bridge', 'hard'): 5773

Отобрано вопросов: 100
Финальное распределение по стратам:
  ('bridge', 'hard'): 50
  ('comparison', 'hard'): 50


## Шаг 4. Формирование корпуса документов

**Состав корпуса:**

1. **Target paragraphs** — все 10 параграфов для каждого из 100 отобранных вопросов (200 gold + 800 distractor = 1000 параграфов).
2. **Background paragraphs** — параграфы из дополнительных `BACKGROUND_QUESTIONS` вопросов (не пересекающихся с target). Они создают реалистичный «фоновый шум» для retriever'а в end-to-end режиме.

**Зачем фон?** Если в базе будут только 1000 параграфов, привязанных к 100 вопросам, retriever будет работать в тепличных условиях. Добавляя 4000+ фоновых параграфов, мы делаем задачу реалистичной: retriever должен найти gold среди тысяч не связанных с вопросом документов.

**Важно:** дедупликация по title — если один и тот же wiki-параграф встречается в нескольких вопросах (как distractor или gold), он попадает в корпус только один раз. Но в gold_mapping сохраняются все связи.

In [ ]:
# Множество ID целевых вопросов
target_ids = {e["id"] for e in selected}

# Отбираем фоновые вопросы из оставшихся
non_target = [e for e in clean_examples if e["id"] not in target_ids]
rng_bg = random.Random(SEED + 1000)
background = rng_bg.sample(non_target, min(BACKGROUND_QUESTIONS, len(non_target)))
print(f"Background вопросов: {len(background)}")

# Собираем корпус: title -> text (дедупликация по title)
corpus = {}  # title -> text
title_sources = defaultdict(set)  # title -> set of question_ids где этот параграф встречается
title_roles = defaultdict(set)  # title -> {"gold", "distractor", "background"}

def add_question_paragraphs(ex, is_target):
    gold_set = set(ex["gold_titles"])
    for title, text in ex["paragraphs"].items():
        if title in corpus:
            # проверяем консистентность (могут быть чуть разные версии)
            pass
        else:
            corpus[title] = text
        title_sources[title].add(ex["id"])
        if is_target:
            if title in gold_set:
                title_roles[title].add("gold")
            else:
                title_roles[title].add("distractor")
        else:
            title_roles[title].add("background")

for ex in selected:
    add_question_paragraphs(ex, is_target=True)
for ex in background:
    add_question_paragraphs(ex, is_target=False)

print(f"\nИтого уникальных параграфов в корпусе: {len(corpus)}")
roles_counter = Counter()
for t, roles in title_roles.items():
    # приоритет ролей: gold > distractor > background
    if "gold" in roles:
        roles_counter["gold"] += 1
    elif "distractor" in roles:
        roles_counter["distractor"] += 1
    else:
        roles_counter["background"] += 1
print(f"По ролям (приоритет gold>distractor>background):")
for role, n in roles_counter.items():
    print(f"  {role}: {n}")

Background вопросов: 400

Итого уникальных параграфов в корпусе: 4913
По ролям (приоритет gold>distractor>background):
  gold: 200
  distractor: 777
  background: 3936


## Шаг 5. Чанкинг параграфов

Большинство параграфов HotpotQA короткие (< 512 токенов), но для единообразия пропускаем через `RecursiveCharacterTextSplitter`. Это:

- гарантирует одинаковый формат в индексе;
- даёт контроль над размером чанков для последующих экспериментов с chunking-стратегиями;
- сохраняет связь чанк → title (а значит и чанк → question) через metadata.

Если параграф помещается в один чанк — он и будет одним чанком. Это нормально.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = []  # список словарей с полями text, title, chunk_id, role
for title, text in tqdm(corpus.items(), desc="Chunking"):
    parts = splitter.split_text(text)
    for i, part in enumerate(parts):
        # определяем приоритетную роль
        roles = title_roles[title]
        if "gold" in roles:
            role = "gold"
        elif "distractor" in roles:
            role = "distractor"
        else:
            role = "background"

        chunk_id = hashlib.md5(f"{title}__{i}".encode()).hexdigest()
        chunks.append({
            "chunk_id": chunk_id,
            "title": title,
            "chunk_index": i,
            "text": part,
            "role": role,
            "source_question_ids": sorted(title_sources[title]),
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка (символов): {sum(len(c['text']) for c in chunks) / len(chunks):.0f}")
print(f"Max длина: {max(len(c['text']) for c in chunks)}")
print(f"Min длина: {min(len(c['text']) for c in chunks)}")

Chunking:   0%|          | 0/4913 [00:00<?, ?it/s]

Всего чанков: 7997
Средняя длина чанка (символов): 342
Max длина: 512
Min длина: 15


## Шаг 6. Построение эмбеддингов и загрузка в Qdrant

**Модель:** `BAAI/bge-small-en-v1.5` через `fastembed` — 384-dim, работает на CPU быстро, низкая стоимость памяти. Это стандарт для лёгких RAG-систем и она достаточно качественная для research.

**Коллекция:** `rag_clean`, метрика — косинусное сходство. Это дефолт для BGE-семейства.

**Payload индексы:** делаем индекс по полю `title` для быстрой фильтрации в controlled-context режиме (когда нужно выдернуть именно gold-документ).

In [ ]:
from fastembed import TextEmbedding

print(f"Инициализация embedding-модели: {EMBEDDING_MODEL}")
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

# Быстрый тест размерности
test_vec = list(embedder.embed(["test"]))[0]
print(f"Размерность вектора: {len(test_vec)}")
assert len(test_vec) == EMBEDDING_DIM, f"Ожидали {EMBEDDING_DIM}, получили {len(test_vec)}"

Инициализация embedding-модели: BAAI/bge-small-en-v1.5


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Размерность вектора: 384


In [ ]:
# Создаём коллекцию
qdrant.create_collection(
    collection_name=COLLECTION_CLEAN,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

# Индекс по title — нужен для фильтрации в controlled-context режиме
qdrant.create_payload_index(
    collection_name=COLLECTION_CLEAN,
    field_name="title",
    field_schema=PayloadSchemaType.KEYWORD,
)
# Индекс по role — для аналитики и future использования
qdrant.create_payload_index(
    collection_name=COLLECTION_CLEAN,
    field_name="role",
    field_schema=PayloadSchemaType.KEYWORD,
)

print(f"Коллекция {COLLECTION_CLEAN} создана")

Коллекция rag_clean создана


In [ ]:
# Построение эмбеддингов батчами и загрузка в Qdrant
BATCH_SIZE = 128
UPLOAD_BATCH = 256

texts = [c["text"] for c in chunks]
print(f"Считаем эмбеддинги для {len(texts)} чанков...")

# fastembed возвращает генератор, материализуем батчами
vectors = []
for vec in tqdm(embedder.embed(texts, batch_size=BATCH_SIZE), total=len(texts), desc="Embedding"):
    vectors.append(vec.tolist())

print(f"Получено векторов: {len(vectors)}")
assert len(vectors) == len(chunks)

Считаем эмбеддинги для 7997 чанков...


Embedding:   0%|          | 0/7997 [00:00<?, ?it/s]

Получено векторов: 7997


In [ ]:
# Загрузка в Qdrant
points = []
for idx, (chunk, vec) in enumerate(zip(chunks, vectors)):
    points.append(PointStruct(
        id=idx,  # простой целочисленный id для упорядоченности
        vector=vec,
        payload={
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "chunk_index": chunk["chunk_index"],
            "text": chunk["text"],
            "role": chunk["role"],
            "source_question_ids": chunk["source_question_ids"],
        }
    ))

for i in tqdm(range(0, len(points), UPLOAD_BATCH), desc="Upload"):
    batch = points[i:i+UPLOAD_BATCH]
    qdrant.upsert(collection_name=COLLECTION_CLEAN, points=batch, wait=True)

# Проверка
info = qdrant.get_collection(COLLECTION_CLEAN)
print(f"\nКоллекция {COLLECTION_CLEAN}:")
print(f"  points_count: {info.points_count}")
print(f"  status: {info.status}")
assert info.points_count == len(chunks)

Upload:   0%|          | 0/32 [00:00<?, ?it/s]


Коллекция rag_clean:
  points_count: 7997
  status: green


## Шаг 7. Санити-проверка retrieval

Прогоняем 5 случайных вопросов через базовый retrieval и проверяем, что gold-документы действительно находятся в топ-10. Это критично: если retrieval не находит gold, все последующие эксперименты бессмысленны.

In [ ]:
rng_check = random.Random(SEED + 9999)
sample_questions = rng_check.sample(selected, 5)

def retrieve(query, top_k=10):
    qvec = list(embedder.embed([query]))[0].tolist()
    hits = qdrant.query_points(
        collection_name=COLLECTION_CLEAN,
        query=qvec,
        limit=top_k,
        with_payload=True,
    ).points
    return hits

recall_at_10 = 0
for ex in sample_questions:
    hits = retrieve(ex["question"], top_k=10)
    retrieved_titles = [h.payload["title"] for h in hits]
    gold_found = sum(1 for g in ex["gold_titles"] if g in retrieved_titles)
    print(f"\nQ: {ex['question'][:80]}")
    print(f"   gold: {ex['gold_titles']}")
    print(f"   top-3 retrieved: {retrieved_titles[:3]}")
    print(f"   gold found in top-10: {gold_found}/2")
    recall_at_10 += gold_found / 2

print(f"\nСредний Recall@10 (gold) на 5 случайных вопросах: {recall_at_10/5:.2f}")
print("(Если < 0.5 — что-то не так с индексацией)")


Q: Josh Trank and Mike Valerio both work in what industry?
   gold: ['Josh Trank', 'Mike Valerio']
   top-3 retrieved: ['Josh Trank', 'Mike Valerio', 'James Valerio']
   gold found in top-10: 2/2

Q: How is the namesake of The Mountbatten Institute related to Elizabeth II?
   gold: ['Mountbatten Institute', 'Louis Mountbatten, 1st Earl Mountbatten of Burma']
   top-3 retrieved: ['Mountbatten Institute', 'Earl of Merioneth', 'Louis Mountbatten, 1st Earl Mountbatten of Burma']
   gold found in top-10: 2/2

Q: Which film was released first, Home on the Range or Pete's Dragon?
   gold: ['Home on the Range (2004 film)', "Pete's Dragon (2016 film)"]
   top-3 retrieved: ["Pete's Dragon (2016 film)", 'Home on the Range (2004 film)', 'Peter Pan (1953 film)']
   gold found in top-10: 2/2

Q: Which magazine eventually had more monthly publications, Vogue or Y'all?
   gold: ['Vogue (magazine)', "Y'all (magazine)"]
   top-3 retrieved: ["Y'all (magazine)", 'Vogue (magazine)', 'Vogue (British magazi

## Шаг 8. Сохранение артефактов

Фиксируем все артефакты эксперимента на диск. Это даёт **воспроизводимость** и позволяет в следующих ноутбуках начинать работу без повторной индексации.

**Что сохраняем:**

- `questions.json` — 100 отобранных вопросов (id, question, answer, type, level, gold_titles).
- `gold_mapping.json` — question_id → список gold_titles и полные тексты gold-параграфов (для controlled-context).
- `distractor_mapping.json` — question_id → список distractor_titles из исходного HotpotQA (для инъекции семантических дистракторов).
- `corpus_meta.json` — статистика корпуса (сколько gold/distractor/background, размер, размер чанков).
- `config.json` — конфигурация эксперимента (seed, модель, параметры).

In [ ]:
# 1. questions.json — основной список вопросов
questions_out = []
for ex in selected:
    questions_out.append({
        "id": ex["id"],
        "question": ex["question"],
        "answer": ex["answer"],
        "type": ex["type"],
        "level": ex["level"],
        "gold_titles": ex["gold_titles"],
    })

with open(ARTIFACTS_DIR / "questions.json", "w", encoding="utf-8") as f:
    json.dump(questions_out, f, ensure_ascii=False, indent=2)
print(f"Сохранено: questions.json ({len(questions_out)} записей)")

# 2. gold_mapping.json — тексты gold-параграфов для controlled-context режима
gold_mapping = {}
for ex in selected:
    gold_mapping[ex["id"]] = {
        "gold_titles": ex["gold_titles"],
        "gold_texts": {t: ex["paragraphs"][t] for t in ex["gold_titles"]},
    }
with open(ARTIFACTS_DIR / "gold_mapping.json", "w", encoding="utf-8") as f:
    json.dump(gold_mapping, f, ensure_ascii=False, indent=2)
print(f"Сохранено: gold_mapping.json")

# 3. distractor_mapping.json — distractor'ы от HotpotQA для каждого вопроса
distractor_mapping = {}
for ex in selected:
    distractors = {t: text for t, text in ex["paragraphs"].items() if t not in ex["gold_titles"]}
    distractor_mapping[ex["id"]] = distractors
with open(ARTIFACTS_DIR / "distractor_mapping.json", "w", encoding="utf-8") as f:
    json.dump(distractor_mapping, f, ensure_ascii=False, indent=2)
print(f"Сохранено: distractor_mapping.json")

# 4. corpus_meta.json
corpus_meta = {
    "n_target_questions": len(selected),
    "n_background_questions": len(background),
    "n_unique_titles": len(corpus),
    "n_chunks": len(chunks),
    "role_distribution": dict(roles_counter),
    "avg_chunk_len_chars": sum(len(c["text"]) for c in chunks) / len(chunks),
    "type_distribution": dict(Counter(e["type"] for e in selected)),
    "level_distribution": dict(Counter(e["level"] for e in selected)),
}
with open(ARTIFACTS_DIR / "corpus_meta.json", "w", encoding="utf-8") as f:
    json.dump(corpus_meta, f, ensure_ascii=False, indent=2)
print(f"Сохранено: corpus_meta.json")

# 5. config.json
config = {
    "seed": SEED,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": EMBEDDING_DIM,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "collection_clean": COLLECTION_CLEAN,
    "qdrant_url": QDRANT_URL,
    "dataset": "hotpotqa/hotpot_qa:distractor:validation",
    "n_questions": N_QUESTIONS,
    "n_background": BACKGROUND_QUESTIONS,
}
with open(ARTIFACTS_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"Сохранено: config.json")

print(f"\nВсе артефакты в: {ARTIFACTS_DIR.absolute()}")
print("Файлы:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")

Сохранено: questions.json (100 записей)
Сохранено: gold_mapping.json
Сохранено: distractor_mapping.json
Сохранено: corpus_meta.json
Сохранено: config.json

Все артефакты в: /content/drive/MyDrive/rag_experiment/artifacts
Файлы:
  config.json: 0.4 KB
  corpus_meta.json: 0.4 KB
  distractor_mapping.json: 467.9 KB
  gold_mapping.json: 100.1 KB
  questions.json: 29.9 KB


## Шаг 9. Итоговая сводка

Подводим итоги, что у нас готово для следующих этапов эксперимента.

In [ ]:
print("=" * 60)
print("БАЗА ЗНАНИЙ ГОТОВА")
print("=" * 60)
print(f"\nQdrant:")
print(f"  URL: {QDRANT_URL}")
print(f"  Коллекция: {COLLECTION_CLEAN}")
print(f"  Точек: {info.points_count}")
print(f"  Метрика: cosine")
print(f"  Размерность: {EMBEDDING_DIM}")
print(f"\nВопросы: {len(selected)}")
print(f"  bridge/comparison: {Counter(e['type'] for e in selected)}")
print(f"  levels: {Counter(e['level'] for e in selected)}")
print(f"\nКорпус:")
print(f"  Уникальных параграфов: {len(corpus)}")
print(f"  Чанков: {len(chunks)}")
print(f"  Gold / distractor / background: {roles_counter['gold']} / {roles_counter['distractor']} / {roles_counter['background']}")
print(f"\nАртефакты сохранены в: {ARTIFACTS_DIR.absolute()}")
print("\nСледующий шаг: ноутбук 02 — генератор шума и формирование")
print("контаминированной коллекции rag_noisy для end-to-end режима.")

БАЗА ЗНАНИЙ ГОТОВА

Qdrant:
  URL: https://92fb2201-8122-4f15-aa86-326522e1fcea.eu-central-1-0.aws.cloud.qdrant.io
  Коллекция: rag_clean
  Точек: 7997
  Метрика: cosine
  Размерность: 384

Вопросы: 100
  bridge/comparison: Counter({'bridge': 50, 'comparison': 50})
  levels: Counter({'hard': 100})

Корпус:
  Уникальных параграфов: 4913
  Чанков: 7997
  Gold / distractor / background: 200 / 777 / 3936

Артефакты сохранены в: /content/drive/MyDrive/rag_experiment/artifacts

Следующий шаг: ноутбук 02 — генератор шума и формирование
контаминированной коллекции rag_noisy для end-to-end режима.
